In [ ]:
import sys
import ast
import numpy as np
import pickle
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
import pandas as pd
import os
import fnmatch
from multiprocessing import Pool

year = 1988

files = {
    1988 : [
        "/rdata/ian/pico/paperRuns/sensitivityAnalysis/pinsga2_20to30_1988_040_04_10", 
        "/rdata/ian/pico/paperRuns/sensitivityAnalysis/pinsga2_20to30_1988_060_02_10",
        "/rdata/ian/pico/paperRuns/sensitivityAnalysis/pinsga2_20to30_1988_060_04_05",
        "/rdata/ian/pico/paperRuns/sensitivityAnalysis/pinsga2_20to30_1988_060_04_10",
        "/rdata/ian/pico/paperRuns/sensitivityAnalysis/pinsga2_20to30_1988_060_04_15",
        "/rdata/ian/pico/paperRuns/sensitivityAnalysis/pinsga2_20to30_1988_060_04_20",
        "/rdata/ian/pico/paperRuns/sensitivityAnalysis/pinsga2_20to30_1988_060_06_10",
        "/rdata/ian/pico/paperRuns/sensitivityAnalysis/pinsga2_20to30_1988_060_08_10",
        "/rdata/ian/pico/paperRuns/sensitivityAnalysis/pinsga2_20to30_1988_060_10_10",
        "/rdata/ian/pico/paperRuns/sensitivityAnalysis/pinsga2_20to30_1988_080_04_10",
        "/rdata/ian/pico/paperRuns/sensitivityAnalysis/pinsga2_20to30_1988_100_04_10",
        "/rdata/ian/pico/paperRuns/sensitivityAnalysis/pinsga2_20to30_1988_120_04_10"
    ],   


}

input_directories = files[year]


In [ ]:

input_directories = [directory.rstrip('/') for directory in input_directories]
#global_pf_file_name = r"/rdata/ian/pico/paperRuns/badDSSATruns/global_pf.pkl"
#
#global_pf = pd.read_pickle(global_pf_file_name)

output_dir = "/rdata/ian/pico/paperRuns/sensitivityAnalysis"
output_file = "%s/global_run_table_%d.pkl" % (output_dir, year)



raw_master_table = {'year':[], 'yield':[], 'irr_total':[], 'front':[], 'irrigation':[], 'run':[], 'gen':[], 'algorithm':[]}


In [ ]:
def parse_directory_name(full_path):

    file_name = full_path.split("/")[-1]
    
    fields = file_name.split("_")

    result = {}
    
    result["algorithm"] = fields[0]
    result["DM_range"] = fields[1]
    result["year"] = int(fields[2])
    result["pop_size"] = int(fields[3])
    result["eta"] = int(fields[4])
    result["tau"] = int(fields[5])

    return result

    

In [ ]:
def parse_file_name(file_name): 

    results = {}

    file_chunks = file_name.split("_")
    results["run"] = int(file_chunks[0][3:9])
    results["gen"] = int(file_chunks[1][3:9])

    return results 
    

In [ ]:
def get_file_names(directory, pattern): 

    matching_files = []
    for filename in os.listdir(directory):
        if fnmatch.fnmatch(filename, pattern):
            matching_files.append(os.path.join(directory, filename))
    
    return matching_files

In [ ]:
def read_files(files):

    all_solutions = None
    
    for (i, file_path) in enumerate(files):

        # Gather info from the file name
        file_name = file_path.split("/")[-1]
        full_dir = "/".join(file_path.split("/")[:-1])

        run_meta = parse_directory_name(full_dir)

        run_meta.update(parse_file_name(file_name) )
        # Get objective data
        current_objs = pd.read_csv(file_path, delimiter=',', names=["yield", "irr_total"])

        # Pull in metadata on the run        
        current_objs["yield"]     = current_objs["yield"] * -1
        current_objs["irr_total"] = current_objs["irr_total"] 
        current_objs["run"]       = int(run_meta["run"])
        current_objs["gen"]       = int(run_meta["gen"])
        current_objs["algorithm"] = run_meta["algorithm"] 
        current_objs["DM_range"]  = run_meta["DM_range"]  
        current_objs["year"]      = run_meta["year"]
        current_objs["pop_size"]  = run_meta["pop_size"]
        current_objs["eta"]  = run_meta["eta"]
        current_objs["tau"]  = run_meta["tau"]
                                    
        

        # Get decision variable data 
        var_file_path = file_path[:-7] + "var.csv"
        current_vars = pd.read_csv(var_file_path, delimiter=',', header=None)

        headers = ["var%s" % header for header in range(current_vars.shape[1])]
        current_vars = current_vars.set_axis(headers, axis=1)
        
        current_objs = pd.concat([current_objs,current_vars], axis=1)
        
        if all_solutions is None:
            all_solutions = current_objs
        else:
            all_solutions = pd.concat([all_solutions, current_objs])

    return all_solutions


### Read and accumulate data
Read each file individually and then add metadata on the run 

In [ ]:
results = None

def process_dir(directory):
#for directory in input_directories:

    dir_meta = parse_directory_name(directory)

    year = dir_meta["year"]
        
    print("Processing year %s for folder %s\n" % (year, directory))

    obj_files = get_file_names(directory, 'run*obj.csv')

    #if results is None: 
    #    results = read_files(obj_files)
    #else: 
    #    current_results = read_files(obj_files)
    #    results = pd.concat([results,current_results])

    result = read_files(obj_files)

    print("Successfully processed %d total items for %s\n" % (result.shape[0], directory))
    return result


results = []
with Pool(12) as p: 
    results.append(p.map(process_dir, input_directories))

#results = pd.concat([i[0] for i in results])
results = pd.concat(results[0])



### Performance metrics
Measuring the performance of each configuration against a baseline 

In [ ]:

print("Picklin' the datar")

results.to_pickle(output_file)

print("Done")